# FORS Final-Only FID Mode (CelebA)
Loops over step counts and logs FID/time at the end of each run.

In [ ]:
# If needed (uncomment):
# !pip install diffusers torch torchvision torchmetrics accelerate tqdm


In [ ]:
import os
import time
from collections import defaultdict
from contextlib import nullcontext

import torch
import numpy as np
import matplotlib.pyplot as plt

from diffusers import UNet2DModel, DDPMScheduler, DDIMScheduler
from tqdm import tqdm

from fors_sampler import FORSSampler, FORSConfig
from utils import (
    sync,
    to_01,
    make_celeba_loader,
    load_or_compute_real_stats,
    metric_num_samples,
    init_fid_metric,
    sample_with_scheduler,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

# Speed knobs
fast_mode = True
use_amp = (device == "cuda")

if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

model_id = "google/ddpm-celebahq-256"

# Default settings (fast_mode overrides below)
batch_size = 32
num_images = 1000
num_real = 2000
step_counts = [10, 20, 50, 100, 200]
fid_feature = 2048

if fast_mode:
    batch_size = 16
    num_images = 128
    num_real = 256
    fid_feature = 64

# FID requires at least 2 samples in each distribution
num_images = max(num_images, 2)
num_real = max(num_real, 2)

seed = 1234
torch.manual_seed(seed)

cache_dir = "./fid_cache"
os.makedirs(cache_dir, exist_ok=True)

def autocast_ctx():
    if use_amp:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


In [ ]:
unet = UNet2DModel.from_pretrained(model_id).to(device)
ddpm_sched = DDPMScheduler.from_pretrained(model_id)
ddim_sched = DDIMScheduler.from_pretrained(model_id)

fors = FORSSampler(
    model=unet,
    scheduler=ddpm_sched,
    config=FORSConfig(B=1.0, max_resample=100),
    device=device,
)

in_channels = unet.config.in_channels
sample_size = unet.config.sample_size
if isinstance(sample_size, int):
    height = width = sample_size
else:
    height, width = sample_size


In [ ]:
# CelebA real images (cached stats)
real_loader = make_celeba_loader(
    root="./data",
    split="test",
    height=height,
    width=width,
    batch_size=batch_size,
    device=device,
)

real_state = load_or_compute_real_stats(
    cache_dir=cache_dir,
    fid_feature=fid_feature,
    num_real=num_real,
    height=height,
    width=width,
    device=device,
    real_loader=real_loader,
)


In [ ]:
results = defaultdict(lambda: defaultdict(list))

for name in ["ddpm", "ddim", "fors"]:
    for steps in step_counts:
        fid = init_fid_metric(
            fid_feature=fid_feature,
            device=device,
            real_state=real_state,
            real_loader=real_loader,
            num_real=num_real,
        )
        total_time = 0.0
        count = 0
        batch_idx = 0
        pbar = tqdm(total=num_images, desc=f"{name} steps={steps}")
        while count < num_images:
            bs = min(batch_size, num_images - count)
            gen = torch.Generator(device=device).manual_seed(seed + batch_idx)
            if name == "ddpm":
                sync()
                t0 = time.perf_counter()
                x = sample_with_scheduler(
                    unet=unet,
                    scheduler=ddpm_sched,
                    num_inference_steps=steps,
                    batch_size=bs,
                    device=device,
                    in_channels=in_channels,
                    height=height,
                    width=width,
                    generator=gen,
                    autocast_ctx=autocast_ctx,
                )
                sync()
                t_sample = time.perf_counter() - t0
            elif name == "ddim":
                sync()
                t0 = time.perf_counter()
                x = sample_with_scheduler(
                    unet=unet,
                    scheduler=ddim_sched,
                    num_inference_steps=steps,
                    batch_size=bs,
                    device=device,
                    in_channels=in_channels,
                    height=height,
                    width=width,
                    generator=gen,
                    autocast_ctx=autocast_ctx,
                    eta=0.0,
                )
                sync()
                t_sample = time.perf_counter() - t0
            else:
                sync()
                t0 = time.perf_counter()
                with autocast_ctx():
                    x = fors.sample(bs, steps, generator=gen)
                sync()
                t_sample = time.perf_counter() - t0

            fid.update(to_01(x), real=False)
            total_time += t_sample
            count += bs
            batch_idx += 1
            pbar.update(bs)
        pbar.close()

        if metric_num_samples(fid, "fake_features_num_samples") < 2:
            raise RuntimeError("Not enough fake samples to compute FID. Increase num_images.")
        fid_score = float(fid.compute().cpu())

        results[name]["steps"].append(steps)
        results[name]["fid"].append(fid_score)
        results[name]["time"].append(total_time)

        print(
            f"steps={steps:>3}  sampler={name:>4}  FID={fid_score:>7.2f}  time={total_time:>7.2f}s"
        )


In [ ]:
plt.figure(figsize=(6, 4))
for name, scores in results.items():
    plt.plot(scores["steps"], scores["fid"], marker="o", label=name)

plt.xlabel("Reverse steps")
plt.ylabel("FID (lower is better)")
plt.title("CelebA: Final-Only FID vs Steps")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
